[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MadatOnLine/reel-bias/blob/main/notebooks/indian_movies_research.ipynb)

# Indian Movies IMDB Research: Thematic Evolution & Character Bias Analysis

This notebook researches Indian cinema over the last 50 years (1975–2025) using IMDB and supplementary datasets.
It explores two primary research questions:
1. How has thematic content in Indian movies evolved over five decades?
2. What biases exist in character naming conventions and role assignments across gender, religion, and regional lines?

---

### License Notice

This notebook uses **IMDB Non-Commercial Datasets** which are provided for **personal and non-commercial use only**.
The data is subject to the terms outlined at https://developer.imdb.com/non-commercial-datasets/.
Do not redistribute or use this data for commercial purposes.

Supplementary datasets from Mendeley, GitHub, and Kaggle are used under their respective licenses.
Please refer to each source for specific terms of use.

### Ethical Disclaimer

**Important:** This notebook performs name-based classification of character names to infer gender, religion, and regional background.
This approach is **inherently imprecise and potentially sensitive**. Name-based heuristics cannot definitively determine
an individual's identity, beliefs, or background.

All results presented in this notebook are **statistical observations**, not definitive claims about individuals or communities.
The classification confidence scores reflect the uncertainty inherent in this methodology.
Readers should interpret findings with appropriate caution and cultural sensitivity.

In [ ]:
# --- Colab Setup (skip if running locally) ---
import sys, os
if 'google.colab' in sys.modules:
    !pip install -q wordcloud kagglehub hypothesis pyarrow
    if not os.path.exists('/content/bias_analysis.py'):
        !git clone --depth 1 https://github.com/MadatOnLine/reel-bias.git /tmp/reel-bias
        !cp /tmp/reel-bias/notebooks/*.py /content/
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')
    os.chdir('/content')
    os.makedirs('data', exist_ok=True)
    print('Colab setup complete.')
else:
    # Running locally — ensure notebooks/ is on sys.path
    _nb_dir = os.path.dirname(os.path.abspath('__file__'))
    if _nb_dir not in sys.path:
        sys.path.insert(0, _nb_dir)

## Setup & Imports

In [ ]:
import os
import json
import logging
import warnings
from pathlib import Path
from dataclasses import dataclass, field

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from wordcloud import WordCloud
from tqdm.notebook import tqdm
import requests
import nltk
try:
    import kagglehub
except ImportError:
    kagglehub = None
import pyarrow

# Local modules
from data_acquisition import DataAcquisition
from data_cleaning import DataCleaner, assign_period_bin, add_period_column
from thematic_analysis import ThematicAnalyzer
from bias_analysis import BiasAnalyzer
from visualization import Visualizer
from models import MovieRecord, CharacterRecord, AnalysisResult

# Kaggle API key — loaded from environment variable, never hardcoded
# Set KAGGLE_KEY in your environment before running this notebook
KAGGLE_KEY = os.environ.get('KAGGLE_KEY')
if KAGGLE_KEY:
    os.environ['KAGGLE_USERNAME'] = os.environ.get('KAGGLE_USERNAME', '')
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY

# Project paths
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_DATA_DIR = Path('data')
NOTEBOOK_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Parquet cache path
PARQUET_CACHE = DATA_DIR / 'master_df.parquet'

# Visualization defaults
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.figsize'] = (12, 6)

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Constants
CURRENT_YEAR = 2025
MIN_YEAR = 1975
PERIOD_LENGTH = 5

logger.info('Setup complete. Data directory: %s', DATA_DIR)

## Data Acquisition

In [ ]:
# Load all datasets using the DataAcquisition module.
# Downloads are cached in the data/ directory to avoid repeated network calls.

acquirer = DataAcquisition(data_dir=str(DATA_DIR))
raw_datasets = acquirer.load_all()

print(f'Loaded {len(raw_datasets)} / 6 data sources:')
for name, df in raw_datasets.items():
    print(f'  {name}: {df.shape[0]:,} rows, {df.shape[1]} columns')

if not raw_datasets:
    raise RuntimeError(
        'No datasets could be loaded. Check network connectivity and data/ directory.'
    )

## Data Cleaning

In [ ]:
# Clean each source dataset and filter to Indian movies in [1975, 2025].

cleaner = DataCleaner()

# --- Clean IMDB basics ---
imdb_basics_clean = pd.DataFrame()
if 'imdb_basics' in raw_datasets:
    imdb_basics_clean = cleaner.clean_imdb_basics(raw_datasets['imdb_basics'])
    imdb_basics_clean = cleaner.filter_indian_movies(imdb_basics_clean)
    print(f'IMDB basics after cleaning & Indian filter: {len(imdb_basics_clean):,} rows')

# --- Clean IMDB principals ---
imdb_principals_clean = pd.DataFrame()
valid_tconsts = set(imdb_basics_clean['tconst']) if not imdb_basics_clean.empty and 'tconst' in imdb_basics_clean.columns else set()
if 'imdb_principals' in raw_datasets:
    imdb_principals_clean = cleaner.clean_imdb_principals(
        raw_datasets['imdb_principals'], valid_tconsts
    )
    print(f'IMDB principals after cleaning: {len(imdb_principals_clean):,} rows')

# --- Clean IMDB names ---
imdb_names_clean = pd.DataFrame()
if 'imdb_names' in raw_datasets:
    imdb_names_clean = cleaner.clean_imdb_names(raw_datasets['imdb_names'])
    print(f'IMDB names after cleaning: {len(imdb_names_clean):,} rows')

# --- Standardize all sources for merging ---
standardized: dict[str, pd.DataFrame] = {}
if not imdb_basics_clean.empty:
    standardized['imdb_basics'] = cleaner.standardize_columns(imdb_basics_clean, 'imdb_basics')
for source_name in ('mendeley', 'github_bollywood', 'kaggle_indian_movies'):
    if source_name in raw_datasets:
        standardized[source_name] = cleaner.standardize_columns(
            raw_datasets[source_name], source_name
        )
        print(f'{source_name} standardized: {len(standardized[source_name]):,} rows')

print(f'\nTotal standardized sources: {len(standardized)}')

## Merging

In [ ]:
# Merge cleaned datasets into a Master DataFrame with parquet caching.
# On subsequent runs, load from parquet if it exists to skip re-processing.

if PARQUET_CACHE.exists():
    print(f'Loading cached Master DataFrame from {PARQUET_CACHE} ...')
    master_df = pd.read_parquet(PARQUET_CACHE)
    print(f'Loaded {len(master_df):,} rows from parquet cache.')
else:
    master_df = cleaner.merge_datasets(standardized)
    # Cache to parquet for subsequent runs
    master_df.to_parquet(PARQUET_CACHE, index=False)
    print(f'Saved Master DataFrame to {PARQUET_CACHE}')

print(f'\nMaster DataFrame shape: {master_df.shape}')
print(f'Columns: {list(master_df.columns)}')
if 'year' in master_df.columns:
    print(f'Year range: {master_df["year"].min():.0f} – {master_df["year"].max():.0f}')
if 'source' in master_df.columns:
    print(f'\nRecords per source:')
    print(master_df['source'].value_counts().to_string())
master_df.head()

## Exploratory Data Analysis (EDA)

In [ ]:
# Basic summary statistics and distribution plots.

print('=== Master DataFrame Summary ===')
print(f'Total movies: {len(master_df):,}')
if 'year' in master_df.columns:
    print(f'Year range: {master_df["year"].min():.0f} – {master_df["year"].max():.0f}')
if 'period' in master_df.columns:
    print(f'Periods covered: {sorted(master_df["period"].dropna().unique())}')
if 'genres' in master_df.columns:
    genre_series = master_df['genres'].dropna()
    # Handle both list and string genres
    def _explode_genres(val):
        if isinstance(val, list):
            return val
        if isinstance(val, str):
            return [g.strip() for g in val.split(',') if g.strip()]
        return []
    all_genres = genre_series.apply(_explode_genres).explode().dropna()
    print(f'Unique genres: {all_genres.nunique()}')
    print(f'Top 10 genres:\n{all_genres.value_counts().head(10).to_string()}')

print('\n=== Numeric Column Statistics ===')
print(master_df.describe())

# Distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

if 'year' in master_df.columns:
    master_df['year'].dropna().astype(int).hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
    axes[0].set_title('Movie Count by Year')
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Count')

if 'rating' in master_df.columns:
    master_df['rating'].dropna().hist(bins=40, ax=axes[1], color='coral', edgecolor='white')
    axes[1].set_title('Rating Distribution')
    axes[1].set_xlabel('Rating')
    axes[1].set_ylabel('Count')

fig.tight_layout()
plt.show()
plt.close(fig)

## Thematic Analysis

In [ ]:
# Genre trends, co-occurrence, topic modeling, keyword/runtime/rating trends.

thematic = ThematicAnalyzer()

# Ensure master_df has a period column
if 'period' not in master_df.columns:
    master_df = add_period_column(master_df)

# --- Genre trends by 5-year period ---
genre_trends = thematic.genre_trends_by_period(master_df)
print('=== Genre Trends by Period ===')
if not genre_trends.empty:
    print(f'{len(genre_trends)} genre-period combinations')
    print(genre_trends.head(15))
else:
    print('No genre trend data available.')

# --- Genre co-occurrence matrix ---
cooccurrence = thematic.genre_cooccurrence_matrix(master_df)
print(f'\n=== Genre Co-occurrence Matrix ({cooccurrence.shape}) ===')
if not cooccurrence.empty:
    print(cooccurrence.iloc[:8, :8])  # Show top-left corner

# --- Topic modeling on plot summaries ---
topic_results = thematic.topic_model_plots(master_df, n_topics=10)
print(f'\n=== Topic Modeling ===')
if topic_results.get('fallback') == 'genre_only':
    print('Insufficient plot text — fell back to genre-only analysis.')
else:
    print(f'Coherence score: {topic_results.get("coherence_score", 0):.4f}')
    for tid, words in topic_results.get('topic_words', {}).items():
        print(f'  Topic {tid}: {", ".join(words[:5])}')

# --- Keyword, runtime, and rating trends ---
kw_trends = thematic.keyword_trends(master_df, top_n=15)
rt_trends = thematic.runtime_trends(master_df)
rat_trends = thematic.rating_trends(master_df)

print(f'\n=== Keyword Trends: {len(kw_trends)} entries ===')
if not kw_trends.empty:
    print(kw_trends.head(10))

print(f'\n=== Runtime Trends ===')
if not rt_trends.empty:
    print(rt_trends)

print(f'\n=== Rating Trends ===')
if not rat_trends.empty:
    print(rat_trends)

## Bias Analysis

In [ ]:
# Character name classification and role distribution bias analysis.

bias = BiasAnalyzer()

# Build a character names Series from IMDB principals (or master_df fallback)
character_names = pd.Series(dtype=str)
characters_source_df = pd.DataFrame()

if not imdb_principals_clean.empty and 'characters' in imdb_principals_clean.columns:
    # Explode character lists into individual names
    chars_exploded = imdb_principals_clean.explode('characters').copy()
    chars_exploded.rename(columns={'characters': 'character_name'}, inplace=True)
    chars_exploded = chars_exploded[chars_exploded['character_name'].notna()]
    chars_exploded = chars_exploded[chars_exploded['character_name'].astype(str).str.strip() != '']
    character_names = chars_exploded['character_name'].reset_index(drop=True)
    characters_source_df = chars_exploded.reset_index(drop=True)
    print(f'Character names extracted from IMDB principals: {len(character_names):,}')
else:
    print('No IMDB principals data available for character analysis.')

# --- Classify character names ---
if not character_names.empty:
    classified_df = bias.classify_character_names(character_names)
    print(f'\n=== Character Classification Summary ===')
    print(f'Total characters classified: {len(classified_df):,}')
    print(f'\nGender distribution:')
    print(classified_df['inferred_gender'].value_counts().to_string())
    print(f'\nReligion distribution:')
    print(classified_df['inferred_religion'].value_counts().to_string())
    print(f'\nRegion distribution:')
    print(classified_df['inferred_region'].value_counts().to_string())
    print(f'\nConfidence stats:')
    print(classified_df['classification_confidence'].describe())

    # Merge classification results with source data for role analysis
    if not characters_source_df.empty:
        for col in classified_df.columns:
            if col != 'name':
                characters_source_df[col] = classified_df[col].values
        # Add role_type (use category as proxy if role_type not available)
        if 'role_type' not in characters_source_df.columns:
            characters_source_df['role_type'] = characters_source_df.get(
                'category', pd.Series(['other'] * len(characters_source_df))
            )
        # Add period from master_df via tconst join
        if 'period' not in characters_source_df.columns and 'tconst' in characters_source_df.columns:
            period_map = master_df.set_index('tconst')['period'].to_dict() if 'tconst' in master_df.columns else {}
            characters_source_df['period'] = characters_source_df['tconst'].map(period_map)
else:
    classified_df = pd.DataFrame()
    print('Skipping character classification — no character names available.')

# --- Role distributions ---
role_by_gender = pd.DataFrame()
role_by_religion = pd.DataFrame()
temporal_bias = pd.DataFrame()
name_freq = pd.DataFrame()
rep_index = pd.DataFrame()

if not characters_source_df.empty and 'role_type' in characters_source_df.columns:
    if 'inferred_gender' in characters_source_df.columns:
        role_by_gender = bias.role_distribution_by_gender(characters_source_df)
        print(f'\n=== Role Distribution by Gender ===')
        print(role_by_gender)

    if 'inferred_religion' in characters_source_df.columns:
        role_by_religion = bias.role_distribution_by_religion(characters_source_df)
        print(f'\n=== Role Distribution by Religion ===')
        print(role_by_religion)

    if 'period' in characters_source_df.columns:
        temporal_bias = bias.temporal_bias_trends(characters_source_df)
        print(f'\n=== Temporal Bias Trends: {len(temporal_bias)} entries ===')
        if not temporal_bias.empty:
            print(temporal_bias.head(10))

    if 'name' in characters_source_df.columns:
        name_freq = bias.name_frequency_analysis(characters_source_df)
        print(f'\n=== Name Frequency: {len(name_freq)} entries ===')

    rep_index = bias.representation_index(characters_source_df)
    if not rep_index.empty:
        print(f'\n=== Representation Index ===')
        print(rep_index.head(15))

## Statistical Tests

In [ ]:
# Chi-square / Fisher's exact tests for independence between
# character attributes and role assignments.

gender_significance = {}
religion_significance = {}

if (not characters_source_df.empty
    and 'role_type' in characters_source_df.columns
    and 'inferred_gender' in characters_source_df.columns):

    # Filter to rows with known attributes for meaningful tests
    gender_test_df = characters_source_df[
        (characters_source_df['inferred_gender'] != 'unknown')
        & (characters_source_df['role_type'].notna())
    ]
    if len(gender_test_df) > 0 and gender_test_df['inferred_gender'].nunique() >= 2 and gender_test_df['role_type'].nunique() >= 2:
        gender_significance = bias.statistical_significance_tests(
            gender_test_df, attribute_col='inferred_gender', role_col='role_type'
        )
        print('=== Gender–Role Independence Test ===')
        print(f'  Test used: {gender_significance.get("test_used", "N/A")}')
        print(f'  Chi-square statistic: {gender_significance.get("chi2_statistic", 0):.4f}')
        print(f'  p-value: {gender_significance.get("p_value", 1):.6f}')
        print(f'  Degrees of freedom: {gender_significance.get("degrees_of_freedom", 0)}')
        print(f'  Effect size (Cramer\'s V): {gender_significance.get("effect_size", 0):.4f}')
        print(f'  Significant (alpha=0.05): {gender_significance.get("significant", False)}')
    else:
        print('Insufficient data for gender–role independence test.')

if (not characters_source_df.empty
    and 'role_type' in characters_source_df.columns
    and 'inferred_religion' in characters_source_df.columns):

    religion_test_df = characters_source_df[
        (characters_source_df['inferred_religion'] != 'unknown')
        & (characters_source_df['role_type'].notna())
    ]
    if len(religion_test_df) > 0 and religion_test_df['inferred_religion'].nunique() >= 2 and religion_test_df['role_type'].nunique() >= 2:
        religion_significance = bias.statistical_significance_tests(
            religion_test_df, attribute_col='inferred_religion', role_col='role_type'
        )
        print(f'\n=== Religion–Role Independence Test ===')
        print(f'  Test used: {religion_significance.get("test_used", "N/A")}')
        print(f'  Chi-square statistic: {religion_significance.get("chi2_statistic", 0):.4f}')
        print(f'  p-value: {religion_significance.get("p_value", 1):.6f}')
        print(f'  Degrees of freedom: {religion_significance.get("degrees_of_freedom", 0)}')
        print(f'  Effect size (Cramer\'s V): {religion_significance.get("effect_size", 0):.4f}')
        print(f'  Significant (alpha=0.05): {religion_significance.get("significant", False)}')
    else:
        print('Insufficient data for religion–role independence test.')

if not gender_significance and not religion_significance:
    print('No statistical tests could be performed — insufficient character data.')

## Visualizations

In [ ]:
# Generate all publication-quality visualizations using the Visualizer module.

viz = Visualizer()

# --- Genre trends stacked area chart ---
fig_genre = viz.plot_genre_trends(genre_trends)
fig_genre.savefig(DATA_DIR / 'genre_trends.png', bbox_inches='tight')
display(fig_genre)

# --- Genre co-occurrence heatmap ---
fig_heatmap = viz.plot_genre_heatmap(cooccurrence)
fig_heatmap.savefig(DATA_DIR / 'genre_cooccurrence.png', bbox_inches='tight')
display(fig_heatmap)

# --- Topic word clouds ---
topic_words = topic_results.get('topic_words', {})
fig_topics = viz.plot_topic_wordclouds(topic_words)
fig_topics.savefig(DATA_DIR / 'topic_wordclouds.png', bbox_inches='tight')
display(fig_topics)

# --- Role distribution by gender ---
if not role_by_gender.empty:
    fig_role_gender = viz.plot_role_distribution(
        role_by_gender, group_by='gender',
        significance_results=gender_significance if gender_significance else None
    )
    fig_role_gender.savefig(DATA_DIR / 'role_by_gender.png', bbox_inches='tight')
    display(fig_role_gender)

# --- Role distribution by religion ---
if not role_by_religion.empty:
    fig_role_religion = viz.plot_role_distribution(
        role_by_religion, group_by='religion',
        significance_results=religion_significance if religion_significance else None
    )
    fig_role_religion.savefig(DATA_DIR / 'role_by_religion.png', bbox_inches='tight')
    display(fig_role_religion)

# --- Temporal bias trends ---
if not temporal_bias.empty:
    fig_temporal = viz.plot_temporal_bias(
        temporal_bias,
        significance_results=gender_significance if gender_significance else None
    )
    fig_temporal.savefig(DATA_DIR / 'temporal_bias.png', bbox_inches='tight')
    display(fig_temporal)

# --- Name frequency chart ---
if not name_freq.empty:
    fig_names = viz.plot_name_frequency(
        name_freq,
        significance_results=gender_significance if gender_significance else None
    )
    fig_names.savefig(DATA_DIR / 'name_frequency.png', bbox_inches='tight')
    display(fig_names)

# --- Summary dashboard ---
fig_dashboard = viz.plot_summary_dashboard({
    'genre_trends': genre_trends,
    'bias_results': gender_significance if gender_significance else religion_significance,
    'temporal_bias': temporal_bias,
})
fig_dashboard.savefig(DATA_DIR / 'summary_dashboard.png', bbox_inches='tight')
display(fig_dashboard)

print('All visualizations generated and saved to', DATA_DIR)

## Conclusions

### Ethical Reminder

All findings below are **statistical observations** derived from name-based heuristics and publicly available datasets.
They do **not** constitute definitive claims about individuals, communities, or the Indian film industry as a whole.
Name-based classification is inherently imprecise — confidence scores reflect this uncertainty.
Readers should interpret these results with cultural sensitivity and appropriate caution.

In [ ]:
# Assemble and print a summary of all findings.

print('=' * 70)
print('RESEARCH SUMMARY: Indian Cinema — Thematic Evolution & Character Bias')
print('=' * 70)

# --- Dataset overview ---
print(f'\n1. DATASET OVERVIEW')
print(f'   Total movies in Master DataFrame: {len(master_df):,}')
if 'year' in master_df.columns:
    print(f'   Year range: {master_df["year"].min():.0f} – {master_df["year"].max():.0f}')
if 'source' in master_df.columns:
    print(f'   Data sources: {master_df["source"].nunique()}')

# --- Thematic findings ---
print(f'\n2. THEMATIC ANALYSIS')
if not genre_trends.empty:
    top_genre = genre_trends.groupby('genre')['count'].sum().idxmax()
    print(f'   Most popular genre overall: {top_genre}')
    periods = sorted(genre_trends['period'].unique())
    if len(periods) >= 2:
        first_period = periods[0]
        last_period = periods[-1]
        first_top = genre_trends[genre_trends['period'] == first_period].iloc[0]['genre'] if len(genre_trends[genre_trends['period'] == first_period]) > 0 else 'N/A'
        last_top = genre_trends[genre_trends['period'] == last_period].iloc[0]['genre'] if len(genre_trends[genre_trends['period'] == last_period]) > 0 else 'N/A'
        print(f'   Top genre in {first_period}: {first_top}')
        print(f'   Top genre in {last_period}: {last_top}')
else:
    print('   No genre trend data available.')

if topic_results.get('fallback') == 'genre_only':
    print('   Topic modeling: skipped (insufficient plot text).')
else:
    print(f'   Topic modeling coherence: {topic_results.get("coherence_score", 0):.4f}')

# --- Bias findings ---
print(f'\n3. CHARACTER BIAS ANALYSIS')
if not classified_df.empty:
    total_classified = len(classified_df)
    known_gender = (classified_df['inferred_gender'] != 'unknown').sum()
    print(f'   Characters classified: {total_classified:,}')
    print(f'   Gender identified: {known_gender:,} ({known_gender/total_classified*100:.1f}%)')
else:
    print('   No character classification data available.')

# --- Statistical test results ---
print(f'\n4. STATISTICAL SIGNIFICANCE')
if gender_significance:
    sig_str = 'YES' if gender_significance.get('significant') else 'NO'
    print(f'   Gender–Role test: p={gender_significance.get("p_value", 1):.6f}, '
          f'V={gender_significance.get("effect_size", 0):.4f}, '
          f'Significant: {sig_str}')
if religion_significance:
    sig_str = 'YES' if religion_significance.get('significant') else 'NO'
    print(f'   Religion–Role test: p={religion_significance.get("p_value", 1):.6f}, '
          f'V={religion_significance.get("effect_size", 0):.4f}, '
          f'Significant: {sig_str}')
if not gender_significance and not religion_significance:
    print('   No statistical tests were performed.')

print(f'\n5. LIMITATIONS')
print('   - Name-based classification is heuristic and inherently imprecise.')
print('   - Results are statistical observations, not definitive claims.')
print('   - Dataset coverage may vary across time periods and regions.')
print('   - IMDB data is for non-commercial use only.')

print('\n' + '=' * 70)
print('Analysis complete. All results are statistical observations.')
print('=' * 70)